# Specific Fighter XGBoost Predictions

This notebook loads the trained XGBoost winner model, looks up each fighter's most recent pre-fight career-stat snapshot from `data_clean/pre_fight_career_stats.csv`, and predicts a matchup between two searched fighters.

The model exposes `predict_proba`, so it can return a percentage-like estimate. Treat it as the model's estimated probability/confidence, not a perfectly calibrated betting line unless you add a calibration step.

In [31]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_clean").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Uncomment exactly one model path to test it.
#MODEL_PATH = PROJECT_ROOT / "winner" / "models" / "xgb_old_diff_schema_old_reshuffle_shared_scaled_120.joblib"
MODEL_PATH = PROJECT_ROOT / "winner" / "models" / "xgb_min_final.joblib"

DATA_PATH = PROJECT_ROOT / "data_clean" / "w_pre_fight_stats.csv"

bundle = joblib.load(MODEL_PATH)
model = bundle["model"]

# Support both the original model bundle and xgb_min_final.joblib.
if "preprocessors" in bundle:
    saved_preprocessors = bundle["preprocessors"]
    preprocessors = {"context": saved_preprocessors.get("context")}
    preprocessors.update({
        key: {
            **prep,
            "r_col": prep["r_column"],
            "b_col": prep["b_column"],
        }
        for key, prep in saved_preprocessors["paired"].items()
    })
    feature_columns = bundle["model_feature_columns"]
else:
    preprocessors = bundle["shared_diff_preprocessors"]
    feature_columns = bundle["feature_columns"]

stats_df = pd.read_csv(DATA_PATH)
stats_df["date"] = pd.to_datetime(stats_df["date"], errors="coerce")
stats_df = stats_df.sort_values("date").reset_index(drop=True)

print(f"Loaded model: {MODEL_PATH}")
print(f"Loaded fight stats: {DATA_PATH}")
print(f"Rows: {len(stats_df):,} | Model features: {len(feature_columns):,}")

Loaded model: e:\Libraries\Documents\GitHub\UFC-Predictions\winner\models\xgb_min_final.joblib
Loaded fight stats: e:\Libraries\Documents\GitHub\UFC-Predictions\data_clean\w_pre_fight_stats.csv
Rows: 8,751 | Model features: 68


In [32]:
def normalize_name(value):
    return " ".join(str(value).strip().lower().split())


def search_fighters(query, limit=25):
    """Return fighter names in the dataset matching a partial search string."""
    query_norm = normalize_name(query)
    names = pd.concat([stats_df["R_fighter"], stats_df["B_fighter"]], ignore_index=True).dropna().astype(str)
    unique_names = pd.Series(names.unique(), name="fighter")

    if query_norm:
        mask = unique_names.map(lambda name: query_norm in normalize_name(name))
        unique_names = unique_names[mask]

    return unique_names.sort_values().head(limit).reset_index(drop=True).to_frame()


def _matching_fighter_rows(fighter_name):
    """Find rows where fighter_name appears as Red or Blue; exact matches are preferred over contains matches."""
    query_norm = normalize_name(fighter_name)
    records = []

    for side in ["R", "B"]:
        name_col = f"{side}_fighter"
        side_names = stats_df[name_col].fillna("").astype(str)
        side_norm = side_names.map(normalize_name)

        exact_mask = side_norm == query_norm
        contains_mask = side_norm.str.contains(query_norm, regex=False, na=False)

        for match_type, mask in [("exact", exact_mask), ("contains", contains_mask & ~exact_mask)]:
            matches = stats_df.loc[mask].copy()
            if matches.empty:
                continue
            matches["matched_side"] = side
            matches["matched_name"] = matches[name_col]
            matches["match_type"] = match_type
            records.append(matches)

    if not records:
        return pd.DataFrame()

    out = pd.concat(records, ignore_index=True)
    out["match_priority"] = out["match_type"].map({"exact": 0, "contains": 1})
    return out.sort_values(["match_priority", "date"], ascending=[True, False]).reset_index(drop=True)


def latest_fighter_profile(fighter_name):
    """Get the most recent stat snapshot for a fighter, regardless of whether they were R or B in that row."""
    matches = _matching_fighter_rows(fighter_name)
    if matches.empty:
        suggestions = search_fighters(fighter_name, limit=10)
        raise ValueError(f"No fighter found for {fighter_name!r}. Suggestions:\n{suggestions.to_string(index=False)}")

    row = matches.iloc[0]
    side = row["matched_side"]
    suffix_stats = {}

    for key, prep in preprocessors.items():
        if key == "context":
            continue
        col = prep[f"{side.lower()}_col"]
        suffix = col[2:]
        suffix_stats[suffix] = row[col] if col in row.index else np.nan

    return {
        "query": fighter_name,
        "matched_name": row["matched_name"],
        "matched_side": side,
        "match_type": row["match_type"],
        "date": row["date"],
        "opponent": row["B_fighter"] if side == "R" else row["R_fighter"],
        "fight_id": row.get("fight_id", None),
        "stats": suffix_stats,
    }


def _numeric_series(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def transform_raw_rows(raw_rows):
    """Apply the same shared-scaled DIFF preprocessing saved with the trained model."""
    df = pd.DataFrame(raw_rows)
    out = pd.DataFrame(index=df.index)

    if "context" in preprocessors:
        context = preprocessors["context"]
        for col in context["columns"]:
            if col not in df.columns:
                df[col] = np.nan

        context_values = df[context["columns"]].apply(pd.to_numeric, errors="coerce")
        out[context["columns"]] = context["scaler"].transform(
            context["imputer"].transform(context_values)
        )

    for key, prep in preprocessors.items():
        if key == "context":
            continue

        r = _numeric_series(df, prep["r_col"]).fillna(prep["median"])
        b = _numeric_series(df, prep["b_col"]).fillna(prep["median"])
        r_scaled = (r - prep["mean"]) / prep["std"]
        b_scaled = (b - prep["mean"]) / prep["std"]
        out[f"DIFF_{key}"] = r_scaled - b_scaled

    return out[feature_columns]


def build_raw_matchup_row(red_profile, blue_profile, title_bout=False):
    raw = {"title_bout": float(bool(title_bout))}

    for key, prep in preprocessors.items():
        if key == "context":
            continue
        suffix = prep["r_col"][2:]
        raw[prep["r_col"]] = red_profile["stats"].get(suffix, np.nan)
        raw[prep["b_col"]] = blue_profile["stats"].get(suffix, np.nan)

    return raw


def predict_ordered_matchup(red_fighter, blue_fighter, title_bout=False):
    """Predict with red_fighter assigned to Red and blue_fighter assigned to Blue."""
    red_profile = latest_fighter_profile(red_fighter)
    blue_profile = latest_fighter_profile(blue_fighter)
    raw_row = build_raw_matchup_row(red_profile, blue_profile, title_bout=title_bout)
    X = transform_raw_rows([raw_row])

    red_probability = float(model.predict_proba(X)[0, 1])
    predicted_winner = red_profile["matched_name"] if red_probability >= 0.5 else blue_profile["matched_name"]

    return {
        "red_fighter": red_profile["matched_name"],
        "blue_fighter": blue_profile["matched_name"],
        "predicted_winner": predicted_winner,
        "red_win_probability": red_probability,
        "blue_win_probability": 1.0 - red_probability,
        "red_profile": red_profile,
        "blue_profile": blue_profile,
        "raw_row": raw_row,
        "X": X,
    }


def compare_fighters(fighter_a, fighter_b, title_bout=False):
    """
    Compare two fighters using both corner assignments.

    The neutralized estimate averages:
    1. P(fighter_a wins when fighter_a is Red and fighter_b is Blue)
    2. P(fighter_a wins when fighter_a is Blue and fighter_b is Red)
    """
    a_red = predict_ordered_matchup(fighter_a, fighter_b, title_bout=title_bout)
    b_red = predict_ordered_matchup(fighter_b, fighter_a, title_bout=title_bout)

    fighter_a_name = a_red["red_fighter"]
    fighter_b_name = a_red["blue_fighter"]
    fighter_a_probability = (a_red["red_win_probability"] + b_red["blue_win_probability"]) / 2.0
    fighter_b_probability = 1.0 - fighter_a_probability
    predicted_winner = fighter_a_name if fighter_a_probability >= 0.5 else fighter_b_name

    summary = pd.DataFrame([
        {
            "fighter": fighter_a_name,
            "estimated_win_percentage": fighter_a_probability * 100,
        },
        {
            "fighter": fighter_b_name,
            "estimated_win_percentage": fighter_b_probability * 100,
        },
    ])

    orientation_checks = pd.DataFrame([
        {
            "red_fighter": a_red["red_fighter"],
            "blue_fighter": a_red["blue_fighter"],
            "red_win_probability": a_red["red_win_probability"],
            "blue_win_probability": a_red["blue_win_probability"],
        },
        {
            "red_fighter": b_red["red_fighter"],
            "blue_fighter": b_red["blue_fighter"],
            "red_win_probability": b_red["red_win_probability"],
            "blue_win_probability": b_red["blue_win_probability"],
        },
    ])

    profile_rows = pd.DataFrame([
        {
            "fighter": a_red["red_profile"]["matched_name"],
            "latest_snapshot_date": a_red["red_profile"]["date"],
            "appeared_as": a_red["red_profile"]["matched_side"],
            "opponent_in_snapshot": a_red["red_profile"]["opponent"],
            "match_type": a_red["red_profile"]["match_type"],
        },
        {
            "fighter": a_red["blue_profile"]["matched_name"],
            "latest_snapshot_date": a_red["blue_profile"]["date"],
            "appeared_as": a_red["blue_profile"]["matched_side"],
            "opponent_in_snapshot": a_red["blue_profile"]["opponent"],
            "match_type": a_red["blue_profile"]["match_type"],
        },
    ])

    print(f"Predicted winner: {predicted_winner}")
    display(summary.style.format({
        "estimated_win_probability": "{:.4f}",
        "estimated_win_percentage": "{:.2f}%",
    }))
    print("Latest stat snapshots used:")
    display(profile_rows)
    print("Corner-orientation checks:")
    display(orientation_checks.style.format({
        "red_win_probability": "{:.4f}",
        "blue_win_probability": "{:.4f}",
    }))

    return {
        "predicted_winner": predicted_winner,
        "summary": summary,
        "profiles": profile_rows,
        "orientation_checks": orientation_checks,
        "fighter_a_as_red": a_red,
        "fighter_b_as_red": b_red,
    }

## Search for fighter names

Use this before comparing fighters if you are not sure how a name appears in the dataset.

In [33]:
search_fighters("gracie")

,fighter
0,Kron Gracie
1,Renzo Gracie
2,Roger Gracie
3,Rolles Gracie
4,Royce Gracie


## Predict a specific matchup

Change the two names below. Partial names work, but exact names are safest if multiple fighters share a last name.

In [45]:
result = compare_fighters(
    "Gamrot ",
    "Salki",
    title_bout=False,
)

Predicted winner: Quillan Salkilld


,fighter,estimated_win_percentage
0,Mateusz Gamrot,29.22%
1,Quillan Salkilld,70.78%


Latest stat snapshots used:


,fighter,latest_snapshot_date,appeared_as,opponent_in_snapshot,match_type
0,Mateusz Gamrot,2026-04-11,R,Esteban Ribovics,contains
1,Quillan Salkilld,2026-05-02,B,Beneil Dariush,contains


Corner-orientation checks:


,red_fighter,blue_fighter,red_win_probability,blue_win_probability
0,Mateusz Gamrot,Quillan Salkilld,0.2936,0.7064
1,Quillan Salkilld,Mateusz Gamrot,0.7093,0.2907
